# Notebook 02_00 — Feature Rankings (compute once, reused by all FS method notebooks)

Training set has **3,397,858 rows**. Running RFE / SHAP directly on this is extremely slow
(this is the root cause of the original `02_feature_selection.ipynb` hanging/crashing).

**Fix**: compute all 6 rankings on a **stratified subsample** of the training set
(`RANKING_SAMPLE_SIZE = 100,000`, same class proportions as full train).
This only affects *which features are chosen* — the actual classifier training in
every downstream notebook still uses the full resampled training set.

Output: `models/fs_rankings.pkl` — a dict of `{method_name: ranked_feature_indices}`.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, RFE
from sklearn.tree import DecisionTreeClassifier
import shap

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)


Imports OK


In [2]:
d = load_data()
X_train, y_train = d['X_train'], d['y_train']
FEATURE_NAMES = d['feature_names']

print(f'Full train shape: {X_train.shape}')

X_sub, y_sub = stratified_subsample(X_train, y_train, RANKING_SAMPLE_SIZE, seed=42)
print(f'Subsample shape : {X_sub.shape}  (used for ranking computation only)')
print(f'Subsample class counts:\n{pd.Series(y_sub).value_counts().sort_index()}')

Full train shape: (3397857, 72)
Subsample shape : (100000, 72)  (used for ranking computation only)
Subsample class counts:
0     3624
1    65343
2      836
3    25019
4      495
5      857
6     3123
7      242
8      275
9      186
Name: count, dtype: int64


## Ranking functions

In [3]:
def rank_correlation(X, y):
    # y is a nominal multi-class id (LabelEncoder output) -- a single Pearson
    # corrcoef(X[:,i], y) would treat the class ids as ordinal/continuous,
    # which is statistically meaningless for >2 nominal classes. Instead,
    # binarize one-vs-rest per class (point-biserial correlation is a valid
    # special case of Pearson r for a continuous x vs. binary y) and take the
    # strongest association across classes per feature.
    classes = np.unique(y)
    Y_bin = np.stack([(y == c).astype(float) for c in classes], axis=1)  # (n, n_classes)
    Xc = X - X.mean(axis=0, keepdims=True)
    Yc = Y_bin - Y_bin.mean(axis=0, keepdims=True)
    cov   = Xc.T @ Yc                                       # (n_feat, n_classes)
    x_std = np.sqrt((Xc ** 2).sum(axis=0))                  # (n_feat,)
    y_std = np.sqrt((Yc ** 2).sum(axis=0))                  # (n_classes,)
    corr  = cov / (x_std[:, None] * y_std[None, :] + 1e-12)
    score = np.nan_to_num(np.abs(corr)).max(axis=1)         # (n_feat,)
    return np.argsort(score)[::-1]


def rank_chisquare(X, y):
    X_nn = MinMaxScaler().fit_transform(X)
    scores, _ = chi2(X_nn, y)
    return np.argsort(np.nan_to_num(scores))[::-1]


def rank_xgb_importance(X, y, seed=42):
    model = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=seed,
                               n_jobs=N_JOBS, verbosity=0, eval_metric='mlogloss')
    model.fit(X, y)
    return np.argsort(model.feature_importances_)[::-1]


def rank_shap(X, y, seed=42):
    n_bg = min(500, len(X))
    idx = np.random.default_rng(seed).choice(len(X), n_bg, replace=False)
    model = xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=seed,
                               n_jobs=N_JOBS, verbosity=0, eval_metric='mlogloss')
    model.fit(X, y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X[idx])
    if isinstance(shap_values, list):
        # Older SHAP API: list of (n_samples, n_features) arrays, one per class.
        mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    elif shap_values.ndim == 3:
        # shap>=0.45-ish API for multiclass: single (n_samples, n_features,
        # n_classes) array instead of a per-class list -- average over both
        # samples (axis 0) and classes (axis 2) to get one score per feature.
        mean_abs = np.abs(shap_values).mean(axis=(0, 2))
    else:
        # Binary/regression: plain (n_samples, n_features) array.
        mean_abs = np.abs(shap_values).mean(axis=0)
    return np.argsort(mean_abs)[::-1]


def rank_rfe(X, y, seed=42):
    estimator = DecisionTreeClassifier(random_state=seed, max_depth=10)
    rfe = RFE(estimator=estimator, n_features_to_select=1, step=1)
    rfe.fit(X, y)
    return np.argsort(rfe.ranking_)


def rank_consensus(ranks_dict):
    n_feat = len(next(iter(ranks_dict.values())))
    avg_rank = np.zeros(n_feat)
    for ranked in ranks_dict.values():
        rank_array = np.empty(n_feat, dtype=int)
        for position, feat_idx in enumerate(ranked):
            rank_array[feat_idx] = position
        avg_rank += rank_array
    avg_rank /= len(ranks_dict)
    return np.argsort(avg_rank)


print('Ranking functions defined.')

Ranking functions defined.


## Compute all rankings (resumable: skip methods already saved)

In [4]:
RANKINGS_PATH = f'{MODELS_DIR}fs_rankings.pkl'

if os.path.exists(RANKINGS_PATH):
    with open(RANKINGS_PATH, 'rb') as f:
        RANKINGS = pickle.load(f)
    print(f'Loaded existing rankings: {list(RANKINGS.keys())}')
else:
    RANKINGS = {}

RANK_FUNCS = {
    'Correlation'   : rank_correlation,
    'ChiSquare'     : rank_chisquare,
    'XGB_Importance': rank_xgb_importance,
    'SHAP'          : rank_shap,
    'RFE'           : rank_rfe,
}

for name, fn in RANK_FUNCS.items():
    if name in RANKINGS:
        print(f'{name}: already computed, skipping.')
        continue
    t0 = time.time()
    RANKINGS[name] = fn(X_sub, y_sub)
    print(f'{name}: done in {time.time()-t0:.1f}s')
    # Save after each method (resumable if this notebook itself crashes)
    with open(RANKINGS_PATH, 'wb') as f:
        pickle.dump(RANKINGS, f)

if 'Consensus' not in RANKINGS:
    RANKINGS['Consensus'] = rank_consensus(
        {k: v for k, v in RANKINGS.items() if k != 'Consensus'}
    )
    with open(RANKINGS_PATH, 'wb') as f:
        pickle.dump(RANKINGS, f)
    print('Consensus: done')

print(f'\nSaved all rankings -> {RANKINGS_PATH}')

Correlation: done in 0.1s
ChiSquare: done in 0.1s


XGB_Importance: done in 2.2s


SHAP: done in 2.2s


RFE: done in 6.1s
Consensus: done

Saved all rankings -> D:/UAV_/models/fs_rankings.pkl


## Inspect Top-16 features per method

In [5]:
for method, ranked in RANKINGS.items():
    top = [FEATURE_NAMES[i] for i in ranked[:16]]
    print(f'{method:<16}: {top}')

Correlation     : ['radiotap.channel.freq', 'wlan_radio.frequency', 'wlan_radio.channel', 'radiotap.length', 'wlan_radio.phy', 'frame.encap_type', 'wlan.fc.subtype', 'ip.version', 'wlan.fc.type', 'wlan_radio.signal_dbm', 'arp.opcode', 'ip.proto', 'data.len', 'frame.len', 'ip.ttl', 'radiotap.channel.flags.ofdm']
ChiSquare       : ['frame.encap_type', 'wlan_radio.frequency', 'radiotap.channel.freq', 'wlan_radio.phy', 'radiotap.length', 'wlan_radio.timestamp', 'radiotap.mactime', 'wlan_radio.end_tsf', 'wlan_radio.start_tsf', 'wlan.fc.subtype', 'radiotap.channel.flags.ofdm', 'data.len', 'arp.opcode', 'wlan.fc.type', 'tcp.flags.push', 'tcp.time_relative']
XGB_Importance  : ['wlan.fixed.reason_code', 'radiotap.channel.freq', 'tcp.time_relative', 'tcp.dstport', 'data.len', 'ip.version', 'udp.dstport', 'radiotap.mactime', 'arp.opcode', 'frame.len', 'wlan.fc.retry', 'frame.encap_type', 'tcp.srcport', 'wlan_radio.duration', 'tcp.analysis.rto_frame', 'tcp.analysis.flags']
SHAP            : ['tcp.